In [2]:
import numpy as np

PATH = "C://Users//andrelas//Desktop//cns_code//GlioTrace2.0//src//GliotraceProject//stacks_exp_148//exp_148_roi_1_stack.npz"
data = np.load(PATH)

green = data["Tstack"]

In [4]:
import numpy as np
import cv2

def optical_flow_farneback(green: np.ndarray):
    """
    green: RGB uint8 time stack.
      Accepts shapes:
        (H, W, T, 3)  or (T, H, W, 3)
    Returns:
      flow: (T-1, H, W, 2) float32 where flow[t] is motion from frame t -> t+1
            flow[...,0]=dx, flow[...,1]=dy in pixels
    """
    if green.dtype != np.uint8:
        raise TypeError("green must be uint8")

    if green.ndim != 4 or green.shape[-1] != 3:
        raise ValueError("Expected an RGB stack with shape (H,W,T,3) or (T,H,W,3)")

    # Normalize to (T,H,W,3)
    if green.shape[0] == 3 and green.shape[-1] != 3:
        # unlikely; ignore
        pass

    if green.shape[0] != green.shape[1] and green.shape[0] != 501:
        # not a reliable check; keep simple
        pass

    # If it looks like (H,W,T,3), move T to front
    if green.shape[0] == green.shape[1] and green.shape[2] != green.shape[0]:
        # (H,W,T,3) -> (T,H,W,3)
        frames = np.moveaxis(green, 2, 0)
    else:
        # assume already (T,H,W,3)
        frames = green

    T, H, W, _ = frames.shape

    # Convert first frame to gray
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)

    flows = np.empty((T - 1, H, W, 2), dtype=np.float32)

    for t in range(1, T):
        gray = cv2.cvtColor(frames[t], cv2.COLOR_RGB2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=0
        )
        flows[t - 1] = flow
        prev_gray = gray

    return flows

import numpy as np
import cv2

def make_flow_side_by_side_video(
    green: np.ndarray,
    out_path: str = "orig_vs_flow.mp4",
    fps: int = 30,
    flow_clip_mag: float | None = None,   # e.g. 10.0 to clamp magnitude for nicer colors
):
    """
    green: RGB uint8 time stack, shape (H,W,T,3) or (T,H,W,3)
    out_path: mp4 path
    Writes: side-by-side video: original | flow-viz
    """
    if green.dtype != np.uint8:
        raise TypeError("green must be uint8")
    if green.ndim != 4 or green.shape[-1] != 3:
        raise ValueError("Expected RGB stack with shape (H,W,T,3) or (T,H,W,3)")

    # Normalize to (T,H,W,3)
    if green.shape[0] != green.shape[1] and green.shape[0] != 3:
        # not a robust test; keep simple
        frames = green
    else:
        # If it's (H,W,T,3), move T to front
        # Heuristic: if axis-2 looks like time (not 3 and not H/W)
        if green.shape[2] != 3 and green.shape[2] != green.shape[0] and green.shape[2] != green.shape[1]:
            frames = np.moveaxis(green, 2, 0)
        else:
            frames = green

    # If still (H,W,T,3) by mistake, fix it
    if frames.shape[0] == frames.shape[1] and frames.shape[0] != frames.shape[2]:
        frames = np.moveaxis(frames, 2, 0)

    T, H, W, _ = frames.shape

    # Video writer (width doubled for side-by-side)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (W * 2, H), True)
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for {out_path}")

    # Optical flow init
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)

    # We'll write T frames: first frame has "no flow yet" -> black flow panel
    zero_flow_panel = np.zeros((H, W, 3), dtype=np.uint8)

    # Helper: flow -> BGR viz
    def flow_to_bgr(flow_xy: np.ndarray) -> np.ndarray:
        # flow_xy: (H,W,2) float32 (dx,dy)
        fx = flow_xy[..., 0]
        fy = flow_xy[..., 1]
        mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=False)

        if flow_clip_mag is not None:
            mag = np.clip(mag, 0, float(flow_clip_mag))

        # Normalize magnitude to 0..255 for Value channel
        mag_norm = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        hsv = np.zeros((H, W, 3), dtype=np.uint8)
        hsv[..., 0] = (ang * (180 / np.pi) / 2).astype(np.uint8)  # 0..180 in OpenCV
        hsv[..., 1] = 255
        hsv[..., 2] = mag_norm.astype(np.uint8)

        return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # Write first frame (no flow yet)
    orig_bgr = cv2.cvtColor(frames[0], cv2.COLOR_RGB2BGR)
    side0 = np.concatenate([orig_bgr, zero_flow_panel], axis=1)
    writer.write(side0)

    # Process subsequent frames
    for t in range(1, T):
        rgb = frames[t]
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=0
        )

        flow_bgr = flow_to_bgr(flow)
        orig_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        side = np.concatenate([orig_bgr, flow_bgr], axis=1)
        writer.write(side)

        prev_gray = gray

    writer.release()
    return out_path

# Usage:
# out = make_flow_side_by_side_video(green, out_path="orig_vs_flow.mp4", fps=30, flow_clip_mag=10.0)
# print("Wrote:", out)


In [12]:
import numpy as np
import cv2

def optical_flow_farneback_gray(green: np.ndarray):
    """
    green: uint8 grayscale time stack, shape (H,W,T) or (T,H,W)
    Returns flow: (T-1, H, W, 2) float32
    """
    if green.dtype != np.uint8:
        raise TypeError("green must be uint8")
    if green.ndim != 3:
        raise ValueError("Expected grayscale stack with shape (H,W,T) or (T,H,W)")

    # Normalize to (T,H,W)
    if green.shape[0] != green.shape[1] and green.shape[0] != 501:
        frames = green
    else:
        # If (H,W,T), move T to front
        frames = np.moveaxis(green, 2, 0) if green.shape[2] != green.shape[0] else green

    # If still (H,W,T), fix it
    if frames.shape[0] == frames.shape[1] and frames.shape[0] != frames.shape[2]:
        frames = np.moveaxis(frames, 2, 0)

    T, H, W = frames.shape

    prev = frames[0]
    flows = np.empty((T - 1, H, W, 2), dtype=np.float32)

    for t in range(1, T):
        nxt = frames[t]
        flow = cv2.calcOpticalFlowFarneback(
            prev, nxt, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=0
        )
        flows[t - 1] = flow
        prev = nxt

    return flows


def make_flow_side_by_side_video_gray(
    green: np.ndarray,
    out_path: str = "orig_vs_flow.mp4",
    fps: int = 30,
    flow_clip_mag: float | None = None,   # e.g. 10.0 to clamp magnitude
):
    """
    green: uint8 grayscale time stack, shape (H,W,T) or (T,H,W)
    Writes side-by-side video: original gray | flow-viz
    """
    if green.dtype != np.uint8:
        raise TypeError("green must be uint8")
    if green.ndim != 3:
        raise ValueError("Expected grayscale stack with shape (H,W,T) or (T,H,W)")

    # Normalize to (T,H,W)
    if green.shape[0] == green.shape[1] and green.shape[2] != green.shape[0]:
        frames = np.moveaxis(green, 2, 0)  # (H,W,T)->(T,H,W)
    else:
        frames = green  # assume (T,H,W)

    if frames.ndim != 3:
        raise ValueError("Internal shape error after axis normalization")

    T, H, W = frames.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (W * 2, H), True)
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for {out_path}")

    def gray_to_bgr(g):
        return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)

    def flow_to_bgr(flow_xy: np.ndarray) -> np.ndarray:
        fx = flow_xy[..., 0]
        fy = flow_xy[..., 1]
        mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=False)

        if flow_clip_mag is not None:
            mag = np.clip(mag, 0, float(flow_clip_mag))

        mag_norm = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        hsv = np.zeros((H, W, 3), dtype=np.uint8)
        hsv[..., 0] = (ang * (180 / np.pi) / 2).astype(np.uint8)
        hsv[..., 1] = 255
        hsv[..., 2] = mag_norm.astype(np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    prev = frames[0]

    # First frame (no flow yet)
    left0 = gray_to_bgr(prev)
    right0 = np.zeros((H, W, 3), dtype=np.uint8)
    writer.write(np.concatenate([left0, right0], axis=1))

    for t in range(1, T):
        nxt = frames[t]

        flow = cv2.calcOpticalFlowFarneback(
            prev, nxt, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=0
        )

        left = gray_to_bgr(nxt)
        right = flow_to_bgr(flow)
        writer.write(np.concatenate([left, right], axis=1))

        prev = nxt

    writer.release()
    return out_path


# Usage:
# flows = optical_flow_farneback_gray(green)  # optional
out = make_flow_side_by_side_video_gray(green, out_path="orig_vs_flow.mp4", fps=6, flow_clip_mag=10.0)
print("Wrote:", out)


Wrote: orig_vs_flow.mp4
